In [1]:
%use kandy

In [2]:
import dev.biserman.planet.utils.UtilityExtensions.signPow

In [191]:
class Species(
    val name: String,
    val baseGrowthFn: (Double, Double) -> Double,
    val baseDeathFn: (Double) -> Double,
    val biomass: Double,
    val carryingCapacityClass: String? = null,
    val canConsume: ((Species) -> Double?)? = null,
) {
    companion object {
        fun producer(
            name: String,
            carryingCapacityClass: String,
            growthRate: Double,
            deathRate: Double,
            biomass: Double
        ) = Species(
            name,
            { population, percentCapacity -> population * growthRate * (1 - percentCapacity) },
            { population -> population * deathRate },
            biomass,
            carryingCapacityClass
        )

        fun consumer(
            name: String,
            growthRate: Double,
            deathRate: Double,
            biomass: Double,
            canConsume: (Species) -> Double?
        ) = Species(
            name,
            { population, percentFood -> population * growthRate * percentFood },
            { population -> population * deathRate },
            biomass,
            null,
            canConsume
        )
    }
}
// I probably need a trophic energy loss function

fun consumeFromMap(map: Map<String, Double>) = { species: Species -> map[species.name] }


In [168]:
fun <T> (Iterable<T>).toPairs(): Sequence<Pair<T, T>> = sequence {
    val list = this@toPairs.toList()
    for (i in 0 until list.size) {
        for (j in 0 until list.size) {
            yield(list[i] to list[j])
        }
    }
}

In [325]:
import kotlin.random.Random

class Ecosystem(var populations: Map<Species, Double>, var carryingCapacities: Map<String, Double>)
fun tickEcosystem(ecosystem: Ecosystem): Ecosystem {
    val attemptedConsumptionIndividuals =
        ecosystem.populations.keys.toPairs()
            .filter { (speciesA, speciesB) -> speciesA.canConsume != null && speciesA.canConsume!!(speciesB) != null }
            .associateWith { (speciesA, speciesB) ->
                min(
                    ecosystem.populations[speciesA]!! * (speciesA.canConsume!!(speciesB) ?: 0.0) / speciesB.biomass,
                    ecosystem.populations[speciesA]!! * ecosystem.populations[speciesB]!! *
                        if (speciesA.name == "hare") 0.001 else if (speciesA.name == "wolf") 0.01 else 1.0
                ) * Random.nextDouble(0.9, 1.1)
            }

    println("desiredConsumption: ${attemptedConsumptionIndividuals.mapKeys { it.key.first.name to it.key.second.name }}")

    val predationImpact = attemptedConsumptionIndividuals.keys
        .groupBy { it.second }
        .mapValues { (_, species) -> species.sumOf { attemptedConsumptionIndividuals[it]!! } }
    val overPredationAdjustment = predationImpact.mapValues { (species, desiredConsumption) ->
        if (desiredConsumption == 0.0) 0.0 else min(1.0, ecosystem.populations[species]!! / desiredConsumption)
    }

    val possibleConsumptionIndividuals = attemptedConsumptionIndividuals
        .mapValues { (pair, desiredConsumption) -> desiredConsumption * overPredationAdjustment[pair.second]!! }

    println("possibleConsumption: ${possibleConsumptionIndividuals.mapKeys { it.key.first.name to it.key.second.name }}")

    val consumptionGrowthFactors = possibleConsumptionIndividuals.keys
        .groupBy { it.first }
        .mapValues { (_, species) ->
            species.sumOf {
                possibleConsumptionIndividuals[it]!! * it.second.biomass / (it.first.canConsume!!(
                    it.second
                )!! * ecosystem.populations[it.first]!! + 1)
            } * 2.0 - 1.0
        }

    println("consumptionGrowthFactors: ${consumptionGrowthFactors.mapKeys { it.key.name }}")

    val growthAmounts = ecosystem.populations.mapValues { (species, population) ->
        species.baseGrowthFn(
            population,
            if (species.carryingCapacityClass != null) {
                (population / ecosystem.carryingCapacities[species.carryingCapacityClass]!!)
            } else (consumptionGrowthFactors[species] ?: 0.0).coerceIn(-1.0, 1.0)
        )
    }
    val naturalDeathAmounts =
        ecosystem.populations.mapValues { (species, population) -> species.baseDeathFn(population) }
    val predationDeathAmounts = possibleConsumptionIndividuals.keys
        .groupBy { it.second }
        .mapValues { (_, species) -> species.sumOf { possibleConsumptionIndividuals[it]!! } }

    println("growth: ${growthAmounts.mapKeys { it.key.name }}")
    println("death: ${naturalDeathAmounts.mapKeys { it.key.name }}")
    println("predation: ${predationDeathAmounts.mapKeys { it.key.name }}")

    return Ecosystem(
        ecosystem.populations.mapValues { (species, population) ->
            maxOf(
                0.0,
                population + growthAmounts[species]!! - naturalDeathAmounts[species]!! - (predationDeathAmounts[species]
                    ?: 0.0)
            )
        },
        ecosystem.carryingCapacities
    )
}

In [335]:
val grass = Species.producer("grass", "carpet", 0.9, 0.03, 2.72)
val hare = Species.consumer("hare", 0.4, 0.017, 4.0, consumeFromMap(mapOf("grass" to 7.25)))
val mouse = Species.consumer("mouse", 0.5, 0.025, 0.03, consumeFromMap(mapOf("grass" to 0.05)))
val wolf = Species.consumer("wolf", 0.1, 0.009, 40.0, consumeFromMap(mapOf("hare" to 126.0)))
val hawk = Species.consumer("hawk", 0.1, 0.017, 1.0, consumeFromMap(mapOf("hare" to 9.0, "mouse" to 9.0)))

val testEcosystem = Ecosystem(
    mapOf(
        grass to 5000,
        hare to 500,
        mouse to 500,
        wolf to 5,
//        hawk to 10
    ).mapValues { (_, value) -> value.toDouble() },
    mapOf("carpet" to 200000.0)
)
val history = (1..1000).scan(testEcosystem) { ecosystem, i ->
    val nextEcosystem = tickEcosystem(ecosystem)
    println("=========================")
    println("step $i")
    nextEcosystem.populations.forEach {
        println("${it.key.name}: ${it.value}")
    }
    println("=========================")
    nextEcosystem
}

desiredConsumption: {(hare, grass)=1292.0657077695807, (mouse, grass)=1233.5344031447562, (wolf, hare)=22.628034671474985, (wolf, mouse)=26.969014119616663}
possibleConsumption: {(hare, grass)=1292.0657077695807, (mouse, grass)=1233.5344031447562, (wolf, hare)=22.628034671474985, (wolf, mouse)=26.969014119616663}
consumptionGrowthFactors: {hare=0.9384548952748264, mouse=0.8506417962237933, wolf=-0.3711943100970947}
growth: {grass=4387.5, hare=187.69097905496528, mouse=170.12835924475866, wolf=-0.18559715504854735}
death: {grass=150.0, hare=8.5, mouse=8.5, wolf=0.045}
predation: {grass=2525.600110914337, hare=22.628034671474985, mouse=26.969014119616663}
step 1
grass: 6711.899889085663
hare: 656.5629443834903
mouse: 634.6593451251421
wolf: 4.769402844951452
desiredConsumption: {(hare, grass)=1594.397760029621, (mouse, grass)=1694.8005042582827, (wolf, hare)=28.38629288291573, (wolf, mouse)=32.80057383042133}
possibleConsumption: {(hare, grass)=1594.397760029621, (mouse, grass)=1694.8005

In [336]:
import org.jetbrains.kotlinx.kandy.dsl.plot
import org.jetbrains.kotlinx.kandy.letsplot.layers.line
import org.jetbrains.kotlinx.kandy.letsplot.layers.points

val logFn = { x: Double -> log(x, 10.0) }
val identityFn = { x: Double -> x }
val chosenFn = logFn

plot {
    listOf(
        grass to Color.GREEN,
        hare to Color.YELLOW,
        mouse to Color.PURPLE,
        wolf to Color.RED,
        hawk to Color.BLUE
    ).filter { (species, _) -> history.any { it.populations.containsKey(species) } }
        .forEach { (species, lineColor) ->
            line {
                x(0..<history.size, name = "time")
                y(history.map { chosenFn(it.populations[species]!!) }, name = "${species.name}_population")
                color = lineColor
            }
        }
}


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="vu3BAh"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"time",
"y":"grass_population"
},
"stat":"identity",
"data":{
"time":[0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,29.0,30.0,31.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,64.0,65.0,66.0,67.0,68.0,69.0,70.0,71.0,72.0,73.0,74.0,75.0,76.0,77.0,78.0,79.0,80.0,81.0,82.0,83.0,84.0,85.0,86.0,87.0,88.0,89.0,90.0,91.0,92.0,93.0,94.0,95.0,96.0,97.0,98.0,99.0,100.0,101.0,102.0,103.0,104.0,105.0,106.0,107.0,108.0,109.0,110.0,111.0,112.0,113.0,114.0,115.0,116.0,117.0,118.0,119.0,120.0,121.0,122.0,123.0,124.0,125.0,126.0,127.0,128.0,129.0,130.0,131.0,132.0,133.0,134.0,135.0,136.0,137.0,138.0,139.0,140.0,141.0,142.0,143.0,144.0,145.0,146.0,147.0,148.0,149.0,150.0,151.0,152.0,153.0,154.0,155.0,156.0,157.0,158.0,159.0,160.0,161.0,162.0,163.0,164.0,165.0,166.0,167.0,168.0,169.0,170.0,171.0,172.0,173.0,174.0,175.0,176.0,177.0,178.0,179.0,180.0,181.0,182.0,183.0,184.0,185.0,186.0,187.0,188.0,189.0,190.0,191.0,192.0,193.0,194.0,195.0,196.0,197.0,198.0,199.0,200.0,201.0,202.0,203.0,204.0,205.0,206.0,207.0,208.0,209.0,210.0,211.0,212.0,213.0,214.0,215.0,216.0,217.0,218.0,219.0,220.0,221.0,222.0,223.0,224.0,225.0,226.0,227.0,228.0,229.0,230.0,231.0,232.0,233.0,234.0,235.0,236.0,237.0,238.0,239.0,240.0,241.0,242.0,243.0,244.0,245.0,246.0,247.0,248.0,249.0,250.0,251.0,252.0,253.0,254.0,255.0,256.0,257.0,258.0,259.0,260.0,261.0,262.0,263.0,264.0,265.0,266.0,267.0,268.0,269.0,270.0,271.0,272.0,273.0,274.0,275.0,276.0,277.0,278.0,279.0,280.0,281.0,282.0,283.0,284.0,285.0,286.0,287.0,288.0,289.0,290.0,291.0,292.0,293.0,294.0,295.0,296.0,297.0,298.0,299.0,300.0,301.0,302.0,303.0,304.0,305.0,306.0,307.0,308.0,309.0,310.0,311.0,312.0,313.0,314.0,315.0,316.0,317.0,318.0,319.0,320.0,321.0,322.0,323.0,324.0,325.0,326.0,327.0,328.0,329.0,330.0,331.0,332.0,333.0,334.0,335.0,336.0,337.0,338.0,339.0,340.0,341.0,342.0,343.0,344.0,345.0,346.0,347.0,348.0,349.0,350.0,351.0,352.0,353.0,354.0,355.0,356.0,357.0,358.0,359.0,360.0,361.0,362.0,363.0,364.0,365.0,366.0,367.0,368.0,369.0,370.0,371.0,372.0,373.0,374.0,375.0,376.0,377.0,378.0,379.0,380.0,381.0,382.0,383.0,384.0,385.0,386.0,387.0,388.0,389.0,390.0,391.0,392.0,393.0,394.0,395.0,396.0,397.0,398.0,399.0,400.0,401.0,402.0,403.0,404.0,405.0,406.0,407.0,408.0,409.0,410.0,411.0,412.0,413.0,414.0,415.0,416.0,417.0,418.0,419.0,420.0,421.0,422.0,423.0,424.0,425.0,426.0,427.0,428.0,429.0,430.0,431.0,432.0,433.0,434.0,435.0,436.0,437.0,438.0,439.0,440.0,441.0,442.0,443.0,444.0,445.0,446.0,447.0,448.0,449.0,450.0,451.0,452.0,453.0,454.0,455.0,456.0,457.0,458.0,459.0,460.0,461.0,462.0,463.0,464.0,465.0,466.0,467.0,468.0,469.0,470.0,471.0,472.0,473.0,474.0,475.0,476.0,477.0,478.0,479.0,480.0,481.0,482.0,483.0,484.0,485.0,486.0,487.0,488.0,489.0,490.0,491.0,492.0,493.0,494.0,495.0,496.0,497.0,498.0,499.0,500.0,501.0,502.0,503.0,504.0,505.0,506.0,507.0,508.0,509.0,510.0,511.0,512.0,513.0,514.0,515.0,516.0,517.0,518.0,519.0,520.0,521.0,522.0,523.0,524.0,525.0,526.0,527.0,528.0,529.0,530.0,531.0,532.0,533.0,53

In [328]:
import org.jetbrains.kotlinx.kandy.dsl.plot
import org.jetbrains.kotlinx.kandy.letsplot.layers.line
import org.jetbrains.kotlinx.kandy.letsplot.layers.points

plot {
    line {
        x(1..<history.size, name = "time")
        y((1..<history.size).map { i -> history[i].populations[grass]!! - history[i - 1].populations[grass]!! }, name = "d/dt_grass_population")
        color = Color.GREEN
    }
    line {
        x(1..<history.size, name = "time")
        y((1..<history.size).map { i -> history[i].populations[hare]!! - history[i - 1].populations[hare]!! }, name = "d/dt_hare_population")
        color = Color.YELLOW
    }
    line {
        x(1..<history.size, name = "time")
        y((1..<history.size).map { i -> history[i].populations[wolf]!! - history[i - 1].populations[wolf]!! }, name = "d/dt_wolf_population")
        color = Color.RED
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="cR3miz"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"time",
"y":"d/dt_grass_population"
},
"stat":"identity",
"data":{
"d/dt_grass_population":[1708.6418701219027,1952.9004742512552,2159.203130044805,2382.27645315259,2677.4670906270367,1253.4446459574974,-1268.2123191406008,-3195.288786941299,-2369.5856079885816,-1816.5936190117973,-1426.8744286756537,-1141.5889436856305,-926.5372633926618,-760.6197414513117,-630.1882989661863,-526.0674228419239,-441.8867985200741,-373.1014160043578,-316.391695573745,-269.28326957233776,-229.8986522388559,-195.24000284993576,-102.90648593693277,21.97213370187069,266.7088269577239,528.3265265976329,799.4249780097412,1030.8097359031071,1685.5551749990263,2548.7992008225765,4140.2591945354925,6150.430728557167,9629.450237330966,15277.608388691791,22360.18633399404,28727.698233577627,31145.081696897527,23948.164998231558,10445.311162998667,-2637.0833219994965,-11759.66763418782,-16340.54730730856,-27939.033689628035,-52626.1715963492,-16916.733746201586,-8614.04693240131,-5375.951220161922,-3693.527020491514,-2688.366449462739,-2033.957306842076,-1582.0686592635066,-1256.3206856542838,-1013.6908999767311,-828.274448092965,-683.6383686046938,-568.9103367039188,-476.64258198904554,-401.5817137321619,-339.9283588306971,-288.87468678648247,-39.396267196576446,343.1332365300391,765.4519224513233,1229.422483757925,2107.500249668643,3637.559557573549,6466.787123954598,10892.006107539635,18060.91546005973,27301.161441702046,36231.70399314116,37375.04739935983,26845.98274443063,11014.772303120786,1607.0900221922784,-852.1015773763356,-2434.1194671054836,-790.9592141121684,-2973.8372277626186,-3432.696907720063,-4695.300007205224,-4167.0222238471615,-9697.533534183283,-10967.082324771472,-14489.897043147153,-24963.436931024087,-32518.183671783845,-33977.13873856596,-12235.99874066688,-6951.069953069535,-4548.956649531632,-3212.3617705461384,-2380.6859351837847,-1824.248580700103,-1432.3860490006846,-1145.6899938371844,-363.02350760662193,2216.1577381620264,5483.291149356937,10357.816497837854,17849.918182300808,28216.58992512664,38678.15874034124,41370.15207289203,29317.965513321076,11454.373306218913,2371.092756219092,341.57740079794894,49.01573176542297,4.2001576184120495,-9.873344605235616,1.2952235057309736,-11.503477417223621,-16.59017937447061,-18.83568121076678,-12.230536667950219,-21.20764534996124,-48.8196444309433,-48.143179337028414,-47.8078767399129,-56.995786419603974,-143.6648655165336,-100.47038942514337,-142.99052375598694,-250.44814444085932,-369.1787900828931,-498.8038000817469,-312.8181285734172,-565.0885782854457,-722.9304685357783,-1338.301250800927,-1281.3349820981384,-2453.585768288205,-2446.3499544578954,-3508.9571025563346,-4646.770644450386,-5659.391623080592,-11497.134893358918,-12907.328184289858,-22926.979598448263,-31002.143915736102,-48352.09387858622,-13328.325680866867,-7373.407786170894,-4765.958122382239,-3340.9285591529515,-2463.8656342301056,-1881.4099359658903,-1473.43486027377,-1176.1714536608179,-952.9042813062206,-781.1485050643005,-646.4462538680264,-539.1251166015036,-452.4973949901214,-381.80837181107245,-323.5958212764381,-275.2858644874766,-157.56986061528983,45.46309165540265,263.368299584067,573.2206936596767,693.1082726702621,794.4507756984244,1264.8726114179462,2007.7988629